In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from torch.utils.data import DataLoader, Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
from PIL import Image, ImageEnhance, ImageDraw
import numpy as np
import random
import os
import cv2
import mediapipe as mp

DATASET_PATH  = "/content/Sleeping_Dataset_Cropped"
SAVE_PATH     = "/content/drive/MyDrive/YOLO/sleeping_final_v4.pth"
TRAIN_PATH    = DATASET_PATH + "/Train"
VAL_PATH      = DATASET_PATH + "/Val"

BATCH_SIZE    = 32
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 25
EARLY_STOP    = 7
IMG_SIZE      = 224

EAR_CLOSED    = 0.20
EAR_OPEN      = 0.28

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


class EARExtractor:

    LEFT_EYE_IDX  = [362, 385, 387, 263, 373, 380]
    RIGHT_EYE_IDX = [33,  160, 158, 133, 153, 144]

    FALLBACK_FEATURE = [0.22, 0.22, 0.22, 0.0, 0.22, 0.22, 0.0]

    def __init__(self):
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh    = self.mp_face_mesh.FaceMesh(
            static_image_mode=True,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.3,
            min_tracking_confidence=0.3
        )

    @staticmethod
    def _ear(landmarks, indices, w, h):
        pts = np.array([[landmarks[i].x * w,
                         landmarks[i].y * h] for i in indices],
                       dtype=np.float32)
        A = np.linalg.norm(pts[1] - pts[5])
        B = np.linalg.norm(pts[2] - pts[4])
        C = np.linalg.norm(pts[0] - pts[3])
        return (A + B) / (2.0 * C + 1e-6)

    def extract(self, pil_img):
        img_rgb = np.array(pil_img.convert("RGB"))
        h, w    = img_rgb.shape[:2]
        results = self.face_mesh.process(img_rgb)

        if not results.multi_face_landmarks:
            return np.array(self.FALLBACK_FEATURE, dtype=np.float32)

        lm = results.multi_face_landmarks[0].landmark

        ear_l = self._ear(lm, self.LEFT_EYE_IDX,  w, h)
        ear_r = self._ear(lm, self.RIGHT_EYE_IDX, w, h)
        ear_m = (ear_l + ear_r) / 2.0
        ear_d = abs(ear_l - ear_r)
        ear_l_n = float(np.clip((ear_l - EAR_CLOSED) /
                                (EAR_OPEN - EAR_CLOSED), 0.0, 1.0))
        ear_r_n = float(np.clip((ear_r - EAR_CLOSED) /
                                (EAR_OPEN - EAR_CLOSED), 0.0, 1.0))

        return np.array(
            [ear_l, ear_r, ear_m, ear_d, ear_l_n, ear_r_n, 1.0],
            dtype=np.float32
        )

    def close(self):
        self.face_mesh.close()


class EARDataset(Dataset):

    def __init__(self, root, transform, extractor: EARExtractor):
        self.base      = datasets.ImageFolder(root, transform=None)
        self.transform = transform
        self.extractor = extractor
        self.class_to_idx = self.base.class_to_idx
        self.samples      = self.base.samples

        print(f"  Pre-extracting EAR features for {len(self.samples)} samples...")
        self.ear_features = []
        for idx, (fpath, _) in enumerate(self.samples):
            pil = Image.open(fpath).convert("RGB")
            self.ear_features.append(self.extractor.extract(pil))
            if (idx + 1) % 500 == 0:
                detected = sum(f[6] > 0.5 for f in self.ear_features)
                print(f"    {idx+1}/{len(self.samples)}  "
                      f"landmarks detected: {detected}/{idx+1} "
                      f"({100*detected/(idx+1):.1f}%)")

        detected = sum(f[6] > 0.5 for f in self.ear_features)
        print(f"  ✅ EAR extraction done. "
              f"Landmarks detected: {detected}/{len(self.samples)} "
              f"({100*detected/len(self.samples):.1f}%)\n")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, label = self.samples[idx]
        pil          = Image.open(fpath).convert("RGB")
        img_tensor   = self.transform(pil)
        ear_tensor   = torch.tensor(self.ear_features[idx], dtype=torch.float32)
        return img_tensor, ear_tensor, label




class SimulateGlasses:

    def __init__(self, p=0.25):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img  = img.convert("RGBA")
        w, h = img.size
        y1   = int(h * 0.28)
        y2   = int(h * 0.50)
        tint_type = random.choices(
            ["frame_only", "tinted", "dark"],
            weights=[0.30, 0.45, 0.25]
        )[0]
        overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
        draw    = ImageDraw.Draw(overlay)
        if tint_type == "frame_only":
            fill = None
        elif tint_type == "tinted":
            tints = [(160,100,30,110),(80,140,80,100),(60,100,180,110),
                     (120,120,120,100),(180,80,30,110)]
            fill = random.choice(tints)
        else:
            fill = (15, 15, 15, 175)
        for lx1, lx2 in [(int(w*0.06), int(w*0.46)),
                         (int(w*0.54), int(w*0.94))]:
            draw.rectangle([lx1, y1, lx2, y2],
                           outline=(40,40,40,255), width=3, fill=fill)
        draw.line([(int(w*0.46),(y1+y2)//2),(int(w*0.54),(y1+y2)//2)],
                  fill=(40,40,40,255), width=2)
        return Image.alpha_composite(img, overlay).convert("RGB")


class SimulateMakeup:
    def __init__(self, p=0.20):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img  = img.copy()
        w, h = img.size
        colors = [(35,15,15),(55,55,55),(45,15,55),
                  (15,15,15),(70,30,10),(20,40,55)]
        color     = random.choice(colors)
        intensity = random.uniform(0.5, 1.0)
        y_top = int(h * 0.28)
        y_bot = int(h * 0.42)
        arr = np.array(img, dtype=np.float32)
        for y in range(y_top, y_bot):
            t     = (y - y_top) / max(y_bot - y_top - 1, 1)
            alpha = intensity * t * 0.75
            for c, cv in enumerate(color):
                arr[y, :, c] = arr[y, :, c] * (1 - alpha) + cv * alpha
        if random.random() < 0.4:
            for y in range(int(h*0.42), int(h*0.48)):
                t     = 1.0 - (y - int(h*0.42)) / max(int(h*0.48)-int(h*0.42)-1, 1)
                alpha = intensity * t * 0.50
                for c, cv in enumerate(color):
                    arr[y, :, c] = arr[y, :, c] * (1 - alpha) + cv * alpha
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


class RandomBackground:
    def __init__(self, p=0.20):
        self.p = p
        self.bg_colors = [
            (70,100,140),(140,160,130),(180,160,140),(90,90,90),
            (200,190,175),(50,60,70),(160,130,110),(100,80,120),
            (180,100,60),(60,100,80),
        ]

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img = img.copy().convert("RGB")
        w, h = img.size
        arr  = np.array(img, dtype=np.float32)
        corners = [arr[0:20,0:20], arr[0:20,w-20:w],
                   arr[h-20:h,0:20], arr[h-20:h,w-20:w]]
        bg_est     = np.mean([c.mean(axis=(0,1)) for c in corners], axis=0)
        corner_std = np.std([c.mean() for c in corners])
        if corner_std > 40:
            return img
        new_bg = np.array(random.choice(self.bg_colors), dtype=np.float32)
        dist   = np.linalg.norm(arr - bg_est, axis=2)
        mask   = dist < 60
        for c in range(3):
            arr[:,:,c] = np.where(mask, new_bg[c], arr[:,:,c])
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


class SimulateSquint:
    def __init__(self, p=0.20): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        img = img.copy(); w, h = img.size
        y1, y2 = int(h*0.28), int(h*0.52)
        strip = ImageEnhance.Brightness(
            img.crop((0,y1,w,y2))).enhance(random.uniform(0.4,0.7))
        img.paste(strip, (0,y1)); return img


class SimulateLowLight:
    def __init__(self, p=0.25): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.3,0.6))
        return ImageEnhance.Contrast(img).enhance(random.uniform(0.5,0.8))


class SimulateHeadDown:
    def __init__(self, p=0.15): self.p = p
    def __call__(self, img):
        if random.random() > self.p: return img
        img = img.copy(); w, h = img.size
        shift   = int(h * random.uniform(0.10, 0.25))
        cropped = img.crop((0, shift, w, h))
        result  = Image.new("RGB", (w, h), (0,0,0))
        result.paste(cropped, (0,0)); return result



train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.70, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=(-10, 25)),
    transforms.ColorJitter(brightness=0.6, contrast=0.6,
                           saturation=0.3, hue=0.05),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomPerspective(distortion_scale=0.4, p=0.4),
    RandomBackground(p=0.20),
    SimulateMakeup(p=0.20),
    SimulateLowLight(p=0.25),
    SimulateGlasses(p=0.25),
    SimulateSquint(p=0.20),
    SimulateHeadDown(p=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


extractor     = EARExtractor()

print("Building training dataset...")
train_dataset = EARDataset(TRAIN_PATH, train_transform, extractor)
print("Building validation dataset...")
val_dataset   = EARDataset(VAL_PATH,   val_transform,   extractor)

idx_to_cls   = {v: k for k, v in train_dataset.class_to_idx.items()}
counts       = Counter(label for _, label in train_dataset.samples)
sleeping_idx = train_dataset.class_to_idx.get(
                   "Sleeping", train_dataset.class_to_idx.get("sleeping", 1))
awake_idx    = 1 - sleeping_idx

print(f"Class mapping : {train_dataset.class_to_idx}")
print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")
print("Train counts  :")
for idx, cnt in sorted(counts.items()):
    print(f"  {idx_to_cls[idx]}: {cnt}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)




labels_list   = [label for _, label in train_dataset.samples]
class_weights = compute_class_weight('balanced',
                                     classes=np.array([0,1]),
                                     y=labels_list)
weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"Class weights : {class_weights}")


class EyeAttention(nn.Module):
    def __init__(self, in_channels=1280, spatial_size=7):
        super().__init__()
        self.spatial_size   = spatial_size
        self.attention_conv = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(128, 1,   kernel_size=1),
        )
        prior       = torch.ones(1, 1, spatial_size, spatial_size)
        row_weights = [0.5, 2.0, 3.0, 3.0, 2.0, 0.5, 0.3]
        for row, w in enumerate(row_weights):
            prior[0, 0, row, :] = w
        self.register_buffer('eye_prior', prior)

    def forward(self, x):
        attn_logits  = self.attention_conv(x) * self.eye_prior
        B            = x.size(0)
        attn_weights = F.softmax(
            attn_logits.view(B, -1), dim=1
        ).view(B, 1, self.spatial_size, self.spatial_size)
        out = (x * attn_weights).mean(dim=[2,3])
        return out, attn_weights



class SleepingModelV4(nn.Module):

    def __init__(self):
        super().__init__()


        backbone      = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
        self.features = backbone.features
        self.eye_attn = EyeAttention(in_channels=1280, spatial_size=7)

        self.ear_proj = nn.Sequential(
            nn.Linear(7, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2),
        )

    def forward(self, x, ear):
        # Appearance stream
        feat           = self.features(x)
        pooled, attn   = self.eye_attn(feat)

        # Geometry stream
        ear_feat       = self.ear_proj(ear)

        # Fusion
        fused          = torch.cat([pooled, ear_feat], dim=1)
        logits         = self.classifier(fused)

        return logits, attn


criterion = nn.CrossEntropyLoss(weight=weights)
model     = SleepingModelV4().to(device)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, ear_feats, labels in loader:
        images    = images.to(device)
        ear_feats = ear_feats.to(device)
        labels    = labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(images, ear_feats)
        loss      = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)


def validate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, ear_feats, labels in loader:
            images    = images.to(device)
            ear_feats = ear_feats.to(device)
            labels    = labels.to(device)
            logits, _ = model(images, ear_feats)
            preds     = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc        = np.mean(all_preds == all_labels)

    s_mask          = (all_labels == sleeping_idx)
    sleeping_recall = float(np.mean(all_preds[s_mask] == sleeping_idx)) \
                      if s_mask.sum() > 0 else 0.0
    a_mask          = (all_preds == awake_idx)
    awake_precision = float(np.mean(all_labels[a_mask] == awake_idx)) \
                      if a_mask.sum() > 0 else 0.0

    return acc, sleeping_recall, awake_precision, all_preds, all_labels


print("\n" + "="*54)
print("  PHASE 1: Warmup — backbone frozen")
print("="*54)

for param in model.features.parameters():
    param.requires_grad = False

optimizer_p1 = optim.Adam(
    list(model.eye_attn.parameters()) +
    list(model.ear_proj.parameters()) +
    list(model.classifier.parameters()),
    lr=3e-4
)

for epoch in range(1, PHASE1_EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, criterion, optimizer_p1)
    acc, s_rec, a_prec, _, _ = validate(model, val_loader)
    print(f"  [{epoch}/{PHASE1_EPOCHS}]  Loss: {loss:.4f}  "
          f"Acc: {acc:.4f}  Sleeping Recall: {s_rec:.4f}  "
          f"Awake Prec: {a_prec:.4f}")

print("\n" + "="*54)
print("  PHASE 2: Full fine-tune (all layers)")
print("="*54)

for param in model.features.parameters():
    param.requires_grad = True

optimizer_p2 = optim.Adam([
    {'params': model.features.parameters(),   'lr': 3e-5},
    {'params': model.eye_attn.parameters(),   'lr': 1e-4},
    {'params': model.ear_proj.parameters(),   'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=PHASE2_EPOCHS, eta_min=1e-6
)

best_sleeping_recall = 0.0
no_improve           = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    loss = train_one_epoch(model, train_loader, criterion, optimizer_p2)
    acc, s_rec, a_prec, preds, labels = validate(model, val_loader)
    scheduler.step()
    current_lr = optimizer_p2.param_groups[0]['lr']

    print(f"  [{epoch:2d}/{PHASE2_EPOCHS}]  Loss: {loss:.4f}  "
          f"Acc: {acc:.4f}  Sleeping Recall: {s_rec:.4f}  "
          f"Awake Prec: {a_prec:.4f}  LR: {current_lr:.2e}")

    if s_rec > best_sleeping_recall:
        best_sleeping_recall = s_rec
        no_improve           = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"    ✅ Saved  (Sleeping Recall: {s_rec:.4f})")
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP:
            print(f"\n  Early stop — no improvement for {EARLY_STOP} epochs.")
            break


# ─────────────────────────────────────────────────
# FINAL REPORT
# ─────────────────────────────────────────────────
print("\n" + "="*54)
print("  FINAL EVALUATION")
print("="*54)

model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
acc, s_rec, a_prec, preds, labels = validate(model, val_loader)

target_names = [idx_to_cls[i] for i in sorted(idx_to_cls)]
print(classification_report(labels, preds, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(labels, preds))
print(f"\n🏆 Best Sleeping Recall : {best_sleeping_recall:.4f}")
print(f"   Saved to             : {SAVE_PATH}")



print("\nGenerating attention visualization...")

import matplotlib.pyplot as plt

model.eval()
val_sleeping_dir = os.path.join(VAL_PATH, "Sleeping")
sample_files     = random.sample(os.listdir(val_sleeping_dir), 4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, fname in enumerate(sample_files):
    fpath   = os.path.join(val_sleeping_dir, fname)
    pil_img = Image.open(fpath).convert("RGB")
    tensor  = val_transform(pil_img).unsqueeze(0).to(device)
    ear_vec = torch.tensor(
        extractor.extract(pil_img), dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logits, attn = model(tensor, ear_vec)
        pred_idx     = logits.argmax(dim=1).item()
        pred_label   = idx_to_cls[pred_idx]
        confidence   = torch.softmax(logits, dim=1)[0, sleeping_idx].item()
        ear_val      = ear_vec[0, 2].item()   # mean EAR
        detected     = ear_vec[0, 6].item() > 0.5

    attn_map = attn[0, 0].cpu().numpy()
    attn_map = (attn_map - attn_map.min()) / \
               (attn_map.max() - attn_map.min() + 1e-8)
    attn_up  = cv2.resize(attn_map, (224, 224),
                          interpolation=cv2.INTER_LINEAR)

    ear_str  = f"EAR={ear_val:.3f}" if detected else "EAR=N/A"
    axes[0][col].imshow(pil_img.resize((224, 224)))
    axes[0][col].set_title(
        f"GT:Sleeping  Pred:{pred_label}({confidence:.2f})\n{ear_str}",
        fontsize=7)
    axes[0][col].axis("off")

    axes[1][col].imshow(pil_img.resize((224, 224)))
    axes[1][col].imshow(attn_up, alpha=0.5, cmap='hot')
    axes[1][col].set_title("Attention\n(bright = focus)", fontsize=8)
    axes[1][col].axis("off")

plt.suptitle("v4 dual-stream — EAR shown above each image", fontsize=10)
plt.tight_layout()
out = SAVE_PATH.replace(".pth", "_attention.png")
plt.savefig(out, dpi=100)
plt.show()
print(f"✅ Attention map saved: {out}")
extractor.close()


